# Lab Exercise 3:

Question: Speech-to-Text Application for Accessibility

Aim:
To develop a Python-based speech-to-text system that converts spoken commands into text in real time,
provides meaningful user feedback, handles errors gracefully, and allows comparison of different
recognition methods.

In [3]:


# Install dependencies
!pip install soundfile vosk SpeechRecognition pydub pandas --quiet
!pip install openai-whisper --quiet

import os
import json
import time
import pandas as pd
import soundfile as sf
import speech_recognition as sr

# Upload audio file (use Colab's upload button)
from google.colab import files
uploaded = files.upload()

# Pick the uploaded file
audio_file = list(uploaded.keys())[0]
print(f"Using audio file: {audio_file}")

# ==============================================
# 1. Google Web Speech Recognition (Online)
# ==============================================
def recognize_with_google(audio_path):
    r = sr.Recognizer()
    with sr.AudioFile(audio_path) as source:
        audio = r.record(source)
    print("Recognizing... (Google Web Speech)")
    try:
        text = r.recognize_google(audio)
        print("Speech successfully converted to text!")
        return {"success": True, "text": text, "error": None}
    except sr.UnknownValueError:
        msg = "Speech Recognition could not understand audio. Please try speaking more clearly."
        return {"success": False, "text": None, "error": msg}
    except sr.RequestError as e:
        msg = f"Google API unavailable: {e}"
        return {"success": False, "text": None, "error": msg}

# ==============================================
# 2. Vosk (Offline)
# ==============================================
!wget -q https://alphacephei.com/vosk/models/vosk-model-small-en-us-0.15.zip
!unzip -q vosk-model-small-en-us-0.15.zip
vosk_model_path = "vosk-model-small-en-us-0.15"

from vosk import Model, KaldiRecognizer

def recognize_with_vosk(audio_path, model_path=vosk_model_path):
    wf = sf.SoundFile(audio_path)
    model = Model(model_path)
    rec = KaldiRecognizer(model, wf.samplerate)

    result_texts = []
    while True:
        data = wf.buffer_read(4000, dtype='int16')
        if not data:
            break
        # ✅ FIX: convert CFFI buffer to bytes
        byte_data = bytes(data)
        if rec.AcceptWaveform(byte_data):
            res = json.loads(rec.Result())
            if res.get("text"):
                result_texts.append(res["text"])

    final = json.loads(rec.FinalResult())
    if final.get("text"):
        result_texts.append(final["text"])

    text = " ".join(result_texts).strip()
    if text == "":
        return {"success": False, "text": None, "error": "Vosk could not understand audio."}
    return {"success": True, "text": text, "error": None}



import whisper

def recognize_with_whisper(audio_path, model_size="small"):
    print(f"Recognizing... (Whisper {model_size})")
    model = whisper.load_model(model_size)
    result = model.transcribe(audio_path)
    text = result.get("text", "").strip()
    if text == "":
        return {"success": False, "text": None, "error": "Whisper could not transcribe."}
    return {"success": True, "text": text, "error": None}

# ==============================================
# Run All Methods and Compare
# ==============================================
results = []

print("\n--- Google Web Speech ---")
res_google = recognize_with_google(audio_file)
results.append({
    "method": "Google Web Speech (Online)",
    "success": res_google["success"],
    "recognized_text": res_google["text"],
    "error": res_google["error"]
})

print("\n--- Vosk ---")
res_vosk = recognize_with_vosk(audio_file)
results.append({
    "method": "Vosk (Offline)",
    "success": res_vosk["success"],
    "recognized_text": res_vosk["text"],
    "error": res_vosk["error"]
})

print("\n--- Whisper ---")
res_whisper = recognize_with_whisper(audio_file, model_size="small")
results.append({
    "method": "Whisper Small (Offline)",
    "success": res_whisper["success"],
    "recognized_text": res_whisper["text"],
    "error": res_whisper["error"]
})

# Show Comparison Table
df = pd.DataFrame(results)
print("\nComparison Results:")
display(df)

Saving lab3sample.wav to lab3sample (2).wav
Using audio file: lab3sample (2).wav
replace vosk-model-small-en-us-0.15/am/final.mdl? [y]es, [n]o, [A]ll, [N]one, [r]ename: A

--- Google Web Speech ---
Recognizing... (Google Web Speech)
Speech successfully converted to text!

--- Vosk ---

--- Whisper ---
Recognizing... (Whisper small)


100%|████████████████████████████████████████| 461M/461M [00:02<00:00, 183MiB/s]
/usr/local/lib/python3.12/dist-packages/whisper/transcribe.py:132: UserWarning: FP16 is not supported on CPU; using FP32 instead
  warnings.warn("FP16 is not supported on CPU; using FP32 instead")



Comparison Results:


,method,success,recognized_text,error
0,Google Web Speech (Online),True,I believe you are just talking nonsense,None
1,Vosk (Offline),True,i believe you're just talking nonsense,None
2,Whisper Small (Offline),True,I believe you're just talking nonsense.,None


Summary in Simple Words

Upload Your Audio

You start by uploading a recording of your voice or any speech (like a .wav file) into Colab.

Convert Speech to Text Using Different Methods

Google Speech API (Online): Sends your audio to Google over the internet and gets back the text. Very accurate, but needs a working internet connection.

Vosk (Offline): Uses a small program that works entirely on the computer (no internet needed) to listen and convert your speech to text. Works fast, but may make mistakes if the speech is unclear.

Whisper (Offline, AI-powered): Uses a smart AI model from OpenAI that can handle accents, background noise, and longer sentences. It works locally, but needs more computer power.

Show Feedback at Each Stage

Prints messages like:

"Recognizing..." while it’s working

"Speech successfully converted to text!" when done

Friendly error messages if it can’t understand the speech

Collect and Compare Results

Saves the text from all three methods in a table so you can see side by side:

Which method understood the speech best

Which one failed or gave errors

How long each took

Optional Whisper Model

You can include Whisper for better accuracy, especially for noisy or unclear audio.

Outcome

At the end, you have a clear comparison of three different speech-to-text methods, ready to be reported in your lab.